<div style="border-left:4px solid #f472b6;padding:2px 0 2px 16px;margin:6px 0 18px;"><div style="font:800 27px/1.15 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;letter-spacing:-0.02em;">NL2SQL <span style="font-weight:500;color:#f472b6;">Optimization</span></div><div style="font:400 15px/1.55 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#71717a;margin-top:5px;">Five variants of the hybrid design, benchmarked and ranked.</div></div>

<div style="font:400 15px/1.65 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#3f3f46;">The hybrid design works. This page asks what the best version of it is — five variants, each changing one thing, measured on the same questions and ranked.</div>

In [ ]:
# The code comes from GitHub. The repository is private, so this needs a
# GITHUB_TOKEN secret (Add-ons -> Secrets).
import os, subprocess, sys
from pathlib import Path

ROOT = Path("/kaggle/working/nl2sql")
if not ROOT.exists():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = "https://github.com/Kirazul/NL2SQL-demo.git".replace("https://", f"https://{token}@")
    subprocess.run(["git", "clone", "--depth", "1", url, str(ROOT)], check=True)

sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("code:", ROOT)

In [ ]:
# Notebook 1 saved the database, the index and the model weights. Reuse them
# instead of rebuilding: this cell is why notebooks 2-5 start in seconds.
import shutil
from pathlib import Path

SETUP = Path("/kaggle/input/nl2sql-1-setup")
if not SETUP.exists():
    raise SystemExit("Run notebook 1 (Setup) first, then add it as an input to this one.")

for name in ("data", "models"):
    source, target = SETUP / name, ROOT / name
    if source.exists() and not target.exists():
        shutil.copytree(source, target)

print("database:", (ROOT / "data" / "eicu.db").exists())
print("index:   ", (ROOT / "data" / "index.db").exists())

In [ ]:
# Keys live in Kaggle secrets, never in the notebook.
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ("GROQ_API_KEY", "OPENROUTER_API_KEY", "LANGSMITH_API_KEY"):
    try:
        os.environ[name] = secrets.get_secret(name)
    except Exception:
        print(f"{name} not set - the steps that need it will say so")

os.environ["LANGSMITH_TRACING"] = "1"
os.environ["LANGSMITH_PROJECT"] = "nl2sql"

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#f472b6;">1.</span> The five variants</div></div>

In [ ]:
from nl2sql.optimize.variants import catalogue

for v in catalogue():
    print(f"  {v['name']:<11} changes {v['changes']:<15} {v['what']}")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#f472b6;">2.</span> How hard is a question?</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Computed from what the local stage already worked out: how many tables it touches, how many values it filters on, whether it asks for a rate or a ranking.</div></div>

In [ ]:
from nl2sql.nlp.understand import understand
from nl2sql.optimize.variants import difficulty, starting_rung

questions = [
    "How many hospitals are there?",
    "How many patients received aspirin?",
    "What is the mortality rate per hospital region for patients over 65?",
]
for question in questions:
    score = difficulty(understand(question))
    print(f"  {score:.2f}  start at the {starting_rung(score):<7} model   {question}")

<div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;">An easy question does not need the most expensive model. This score decides where to start; perplexity decides whether to climb.</div>

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#f472b6;">3.</span> Perplexity</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">How surprised a model was by its own answer. Low means confident, high means it was guessing — the only quality signal available without knowing the right answer.</div></div>

In [ ]:
from nl2sql.core import graph

state = graph.run('How many patients over 65 received aspirin?', variant="cascade", write=False)
print("difficulty      :", state.get("difficulty"))
print("model that answered:", state.get("cloud_target"))
print("perplexity      :", state.get("perplexity"), "(lower is more confident)")
print("had to climb    :", state.get("escalated"))
print("calls           :", state.get("cloud_calls"))

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#f472b6;">4.</span> Agreement</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">The consensus variant asks three times and keeps the answer the queries agree on, judged on what they return rather than on how they are written.</div></div>

In [ ]:
state = graph.run('How many patients over 65 received aspirin?', variant="consensus", write=False)
for candidate in state.get("candidates", []):
    mark = "kept" if candidate["chosen"] else "    "
    print(f"  {mark}  perplexity {candidate['perplexity']}  {candidate['sql'][:74]}")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#f472b6;">5.</span> The benchmark</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Every variant over the same questions. Accuracy is measured against a reference query where one exists, and against what the variants agree on where none does.</div></div>

In [ ]:
from nl2sql.optimize.benchmark import compare

report = compare(limit=25)     # raise or remove for the full set

In [ ]:
header = list(report["table"][0])
print(" | ".join(f"{h:>16}" for h in header))
print("-" * (19 * len(header)))
for row in report["table"]:
    print(" | ".join(f"{str(row[h]):>16}" for h in header))

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#f472b6;">6.</span> The ranking</div></div>

In [ ]:
for position, (name, accuracy) in enumerate(report["ranking"], 1):
    print(f"  {position}. {name:<11} {accuracy:.0%}")

print("\nwhere each one failed:")
for variant, failures in report["failures"].items():
    if failures:
        print(f"  {variant:<11} {failures}")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#f472b6;">7.</span> Where to put the escalation threshold</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Not a preference: the point that best separates the answers that turned out right from the ones that turned out wrong.</div></div>

In [ ]:
from nl2sql.optimize.benchmark import calibrate
import json

print(json.dumps(calibrate(), indent=1))